# ANKA-50M — R3/M3 Deep Evaluation

> **Status:** Read-only mechanism and benchmark analysis

## Purpose

Evaluate completed R3 and M3 checkpoints using matched validation tokens, mechanism ablations, zero-shot tasks, generation diagnostics, and paired significance tests.

## Execution guidance

This notebook does not train or overwrite checkpoints. Start with the smoke configuration, verify all paths and causal checks, and enable the full evaluation only afterward.

## Primary outputs

Loss/perplexity, ablations, router statistics, benchmark scores, plots, blind-review files, and JSON reports.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Configure the read-only evaluation run

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 1 — Kurulum, Drive and experiment settings
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "tokenizers>=0.15",
    "datasets>=2.19",
    "transformers>=4.40",
    "pandas>=2.0",
    "seaborn>=0.13",
    "scikit-learn>=1.3",
    "scipy>=1.10",
    "tabulate>=0.9",
])

from google.colab import drive
drive.mount("/content/drive")

from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Optional
import gc
import hashlib
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest
from sklearn.metrics import normalized_mutual_info_score
from tokenizers import Tokenizer

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Select a Colab GPU runtime for this evaluation.")

# Keep this False for a smoke run; set it to True only after every section passes.
FULL_EVAL = True

VAL_BATCH_SIZE = 8
VAL_BATCHES = 100 if FULL_EVAL else 20
ABLATION_BATCHES = 40 if FULL_EVAL else 10
EXTERNAL_LIMIT = 1_000 if FULL_EVAL else 150
BLIMP_PER_CONFIG = 50 if FULL_EVAL else 10
CONTROLLED_PER_TASK = 200 if FULL_EVAL else 40
SCORE_BATCH_SIZE = 16
GENERATION_MAX_NEW = 64 if FULL_EVAL else 32

print("GPU        :", torch.cuda.get_device_name(0))
print("VRAM       :", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")
print("FULL_EVAL  :", FULL_EVAL)


## 2. Discover checkpoints and persistent paths

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 2 — Discover paths and checkpoints
DRIVE_RUN = Path("/content/drive/MyDrive/ANKA/run_01")
TOKENIZER_PATH = DRIVE_RUN / "tokenizer/tokenizer.json"
ARCH_ROOT = DRIVE_RUN / "serious_architectures"
REPORT_ROOT = ARCH_ROOT / "deep_evaluation_full"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATHS = {
    "R3": ARCH_ROOT / "R3/final.pt",
    "M3": ARCH_ROOT / "M3/final.pt",
}

MANIFEST_CANDIDATES = [
    Path("/content/anka_shards/manifest.json"),
    DRIVE_RUN / "shards/manifest.json",
    DRIVE_RUN / "tokenized/manifest.json",
]
MANIFEST_PATH = next((p for p in MANIFEST_CANDIDATES if p.exists()), None)

assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"
for arm, path in CHECKPOINT_PATHS.items():
    assert path.exists(), f"{arm} final checkpoint not found: {path}"

print("Tokenizer   :", TOKENIZER_PATH)
print("R3          :", CHECKPOINT_PATHS["R3"])
print("M3          :", CHECKPOINT_PATHS["M3"])
print("Manifest    :", MANIFEST_PATH)
print("report path  :", REPORT_ROOT)

if MANIFEST_PATH is None:
    print("\nUYARI: Token shard manifesti not found.")
    print("Corpus validation and mechanism-loss ablations will-be-skipped;")
    print("controlled and external benchmark tests can still run.")


## 3. Define unchanged shared model components

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 3 — Shared model components used during training (unchanged)
@dataclass
class ANKAConfig:
    name: str
    vocab_size: int = 15_360
    max_seq_len: int = 2_048
    d_model: int = 448
    n_layers: int = 20
    n_heads: int = 7
    n_kv_heads: int = 1
    d_ff: int = 1_216
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0


@dataclass
class ANKAOutput:
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(dim=-1, keepdim=True) + self.eps)
        return y.to(dtype) * self.weight.to(dtype=dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, theta: float):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        pos = torch.arange(max_seq_len).float()
        freq = torch.outer(pos, inv)
        self.register_buffer("cos", freq.cos(), persistent=False)
        self.register_buffer("sin", freq.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq = x.shape[-2]
        cos = self.cos[:seq].to(device=x.device, dtype=x.dtype)[None, None, :, :]
        sin = self.sin[:seq].to(device=x.device, dtype=x.dtype)[None, None, :, :]
        even, odd = x[..., 0::2], x[..., 1::2]
        out = torch.empty_like(x)
        out[..., 0::2] = even * cos - odd * sin
        out[..., 1::2] = even * sin + odd * cos
        return out


class GQAttention(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.dropout = cfg.dropout
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, cfg.max_seq_len, cfg.rope_theta)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, _ = x.shape
        q = self.q_proj(x).view(b, s, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, s, -1))


class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))


def causal_lm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(
        logits[:, :-1].contiguous().view(-1, logits.size(-1)),
        labels[:, 1:].contiguous().view(-1),
    )


def cuda_autocast_enabled() -> bool:
    try:
        return torch.is_autocast_enabled("cuda")
    except TypeError:
        return torch.is_autocast_enabled()


def initialize_model(model: nn.Module, std: float = 0.02) -> None:
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=std)


## 4. Define the trained R3 architecture

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 4 — R3: ANKA Reflective Loop (training-time definition)
class ReflectiveBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(
        self,
        x: torch.Tensor,
        reflection: Optional[torch.Tensor],
        reflection_gate: Optional[torch.Tensor],
        residual_scale: float,
    ) -> torch.Tensor:
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.attn(self.attn_norm(conditioned))
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.ffn(self.ffn_norm(conditioned))
        return x


class ANKAReflectiveLoop(nn.Module):
    def __init__(self, cfg: ANKAConfig, draft_weight: float = 0.25, aux_stride: int = 8):
        super().__init__()
        self.config = cfg
        self.draft_weight = draft_weight
        self.aux_stride = aux_stride
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([ReflectiveBlock(cfg) for _ in range(cfg.n_layers)])
        self.round_embedding = nn.Parameter(torch.empty(2, cfg.d_model))
        self.reflect_in_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_delta_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.reflect_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_gate_logits = nn.Parameter(torch.full((cfg.n_layers,), -2.1972246))
        self.draft_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)
        self.last_metrics = {}

        initialize_model(self)
        nn.init.normal_(self.round_embedding, std=0.002)
        nn.init.normal_(self.reflect_proj.weight, std=0.01)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence exceeds max_seq_len.")

        token_state = self.embedding(input_ids)
        if token_state.is_cuda and cuda_autocast_enabled():
            token_state = token_state.to(torch.bfloat16)
        h = token_state + self.round_embedding[0].to(token_state.dtype)
        for block in self.blocks:
            h = block(h, None, None, self.residual_scale)
        draft = h

        discrepancy = self.reflect_delta_norm(draft - token_state)
        reflection = torch.tanh(self.reflect_proj(torch.cat([
            self.reflect_in_norm(draft), discrepancy
        ], dim=-1)))
        reflection = self.reflect_norm(reflection)

        h = draft + self.round_embedding[1].to(draft.dtype)
        gates = torch.sigmoid(self.reflect_gate_logits)
        for layer_id, block in enumerate(self.blocks):
            gate = gates[layer_id].to(h.dtype)
            h = block(h, reflection, gate, self.residual_scale)

        logits = F.linear(self.final_norm(h), self.embedding.weight)
        loss = None
        final_loss = None
        draft_loss = None
        if labels is not None:
            final_loss = causal_lm_loss(logits, labels)
            if self.training and self.draft_weight > 0:
                draft_hidden = self.draft_norm(draft)[:, :-1:self.aux_stride]
                draft_targets = labels[:, 1::self.aux_stride]
                draft_logits = F.linear(draft_hidden, self.embedding.weight)
                draft_loss = F.cross_entropy(
                    draft_logits.reshape(-1, draft_logits.size(-1)),
                    draft_targets.reshape(-1),
                )
                loss = final_loss + self.draft_weight * draft_loss
            else:
                loss = final_loss

        self.last_metrics = {
            "final_loss": None if final_loss is None else final_loss.detach(),
            "draft_loss": None if draft_loss is None else draft_loss.detach(),
            "reflect_gate": gates.mean().detach(),
            "reflection_rms": reflection.float().square().mean().sqrt().detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_r3() -> ANKAReflectiveLoop:
    cfg = ANKAConfig(
        name="R3",
        d_model=512,
        n_layers=13,
        n_heads=8,
        n_kv_heads=2,
        d_ff=1536,
    )
    return ANKAReflectiveLoop(cfg)


## 5. Define the trained M3 architecture

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 5 — M3: ANKA Sparse Morphing-FFN (training-time definition)
class SparseMorphFFN(nn.Module):
    def __init__(self, d_model: int, n_experts: int):
        super().__init__()
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.n_experts = n_experts
        self.last_balance = None
        self.last_entropy = None
        self.last_load = None

    def forward(self, x: torch.Tensor, experts: nn.ModuleList) -> torch.Tensor:
        shape = x.shape
        flat = x.reshape(-1, shape[-1])
        router_logits = self.router(flat).float()
        probs = torch.softmax(router_logits, dim=-1)
        choice = probs.argmax(dim=-1)
        out = torch.zeros_like(flat)

        for expert_id, expert in enumerate(experts):
            indices = torch.nonzero(choice == expert_id, as_tuple=False).flatten()
            if indices.numel() == 0:
                continue
            selected = flat.index_select(0, indices)
            expert_out = expert(selected)
            p = probs.index_select(0, indices)[:, expert_id].to(expert_out.dtype)
            straight_through = 1.0 + p - p.detach()
            dispatched = expert_out * straight_through.unsqueeze(-1)
            out = out.index_copy(0, indices, dispatched.to(dtype=out.dtype))

        importance = probs.mean(dim=0)
        load = F.one_hot(choice, self.n_experts).float().mean(dim=0)
        self.last_balance = self.n_experts * torch.sum(importance * load)
        self.last_entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()
        self.last_load = load.detach()
        return out.view(shape)


class MorphBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig, stage_experts: nn.ModuleList):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.morph = SparseMorphFFN(cfg.d_model, len(stage_experts))
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)

    def forward(self, x: torch.Tensor, stage_experts: nn.ModuleList) -> torch.Tensor:
        x = x + self.residual_scale * self.attn(self.attn_norm(x))
        x = x + self.residual_scale * self.morph(self.ffn_norm(x), stage_experts)
        return x


class ANKASparseMorph(nn.Module):
    def __init__(self, cfg: ANKAConfig, n_stages: int = 5, experts_per_stage: int = 4):
        super().__init__()
        assert cfg.n_layers % n_stages == 0
        self.config = cfg
        self.n_stages = n_stages
        self.experts_per_stage = experts_per_stage
        self.layers_per_stage = cfg.n_layers // n_stages
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.stage_experts = nn.ModuleList([
            nn.ModuleList([
                SwiGLU(cfg.d_model, cfg.d_ff)
                for _ in range(experts_per_stage)
            ])
            for _ in range(n_stages)
        ])
        self.blocks = nn.ModuleList([
            MorphBlock(cfg, self.stage_experts[layer_id // self.layers_per_stage])
            for layer_id in range(cfg.n_layers)
        ])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.balance_weight = 0.01
        self.last_metrics = {}
        initialize_model(self)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence exceeds max_seq_len.")
        h = self.embedding(input_ids)
        if h.is_cuda and cuda_autocast_enabled():
            h = h.to(torch.bfloat16)
        balances, entropies = [], []
        stage_loads = [[] for _ in range(self.n_stages)]
        for layer_id, block in enumerate(self.blocks):
            stage_id = layer_id // self.layers_per_stage
            h = block(h, self.stage_experts[stage_id])
            balances.append(block.morph.last_balance)
            entropies.append(block.morph.last_entropy)
            stage_loads[stage_id].append(block.morph.last_load)

        logits = F.linear(self.final_norm(h), self.embedding.weight)
        balance_loss = torch.stack(balances).mean()
        lm_loss = causal_lm_loss(logits, labels) if labels is not None else None
        loss = None if lm_loss is None else lm_loss + self.balance_weight * balance_loss
        mean_loads = [torch.stack(items).mean(0) for items in stage_loads]
        self.last_metrics = {
            "lm_loss": None if lm_loss is None else lm_loss.detach(),
            "balance_loss": balance_loss.detach(),
            "router_entropy": torch.stack(entropies).mean().detach(),
            "stage_loads": torch.stack(mean_loads).detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_m3() -> ANKASparseMorph:
    cfg = ANKAConfig(
        name="M3",
        d_model=448,
        n_layers=20,
        n_heads=7,
        n_kv_heads=1,
        d_ff=1216,
    )
    return ANKASparseMorph(cfg, n_stages=5, experts_per_stage=4)


## 6. Load the tokenizer and checkpoints safely

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 6 — Tokenizer, safe model loading, and checkpoint inventory
TOKENIZER = Tokenizer.from_file(str(TOKENIZER_PATH))
PAD_ID = TOKENIZER.token_to_id("<pad>")
BOS_ID = TOKENIZER.token_to_id("<empty>")
EOS_ID = TOKENIZER.token_to_id("<eos>")
PAD_ID = 0 if PAD_ID is None else PAD_ID
BOS_ID = 1 if BOS_ID is None else BOS_ID

BUILDERS = {"R3": build_r3, "M3": build_m3}


def release_memory(*objects) -> None:
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_model(arm: str):
    checkpoint = torch.load(
        CHECKPOINT_PATHS[arm],
        map_location="cpu",
        weights_only=False,
    )
    model = BUILDERS[arm]()
    missing, unexpected = model.load_state_dict(checkpoint["model"], strict=False)
    assert not missing, f"{arm} missing state keys: {missing[:10]}"
    assert not unexpected, f"{arm} extra state keys: {unexpected[:10]}"
    model = model.to(DEVICE).eval()
    return model, checkpoint


inventory = []
for arm in ("R3", "M3"):
    model, checkpoint = load_model(arm)
    parameters = model.num_trainable_parameters()
    assert parameters < 50_000_000
    assert checkpoint.get("arm", arm) == arm
    inventory.append({
        "arm": arm,
        "parameters": parameters,
        "seen_tokens": int(checkpoint.get("seen_tokens", -1)),
        "step": int(checkpoint.get("step", -1)),
        "history_records": len(checkpoint.get("history", [])),
        "checkpoint_gib": CHECKPOINT_PATHS[arm].stat().st_size / 2**30,
    })
    del model, checkpoint
    gc.collect()
    torch.cuda.empty_cache()

inventory_df = pd.DataFrame(inventory)
display(inventory_df)
inventory_df.to_csv(REPORT_ROOT / "checkpoint_inventory.csv", index=False)


## 7. Run causal-leakage tests

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 7 — Causal leakage testi
# M3 sparse dispatch'te suffix when-changed expert matrislerinin batch size da
# can change. In this case TF32 may create small rounding differences for the same prefix.
# Therefore pass/fail is decided using the true future-token gradient test.

def causal_test(
    arm: str,
    prefix_len: int = 17,
    total_len: int = 32,
) -> dict:
    model, _ = load_model(arm)
    model = model.float().eval()

    torch.manual_seed(SEED)

    tokens = torch.randint(
        4,
        model.config.vocab_size,
        (1, total_len),
        device=DEVICE,
    )

    # ---------------------------------------------------------
    # 1. Normal forward difference
    # only numerical hassasiyet olcumudur.
    # ---------------------------------------------------------
    changed = tokens.clone()

    changed[:, prefix_len:] = torch.randint(
        4,
        model.config.vocab_size,
        changed[:, prefix_len:].shape,
        device=DEVICE,
    )

    with (
        torch.inference_mode(),
        torch.autocast("cuda", enabled=False),
    ):
        logits_a = model(tokens).logits[:, :prefix_len].float()
        logits_b = model(changed).logits[:, :prefix_len].float()

    forward_max_abs = float(
        (logits_a - logits_b).abs().max().item()
    )

    # ---------------------------------------------------------
    # 2. actual causal check
    # Test whether a prefix logit depends on future embedding activations.
    # gradienti full as zero must-be.
    # ---------------------------------------------------------
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    model.embedding.weight.requires_grad_(True)

    captured = {}

    def capture_embedding(
        _module,
        _inputs,
        output,
    ):
        output.retain_grad()
        captured["output"] = output

    hook = model.embedding.register_forward_hook(
        capture_embedding
    )

    model.zero_grad(set_to_none=True)

    with (
        torch.enable_grad(),
        torch.autocast("cuda", enabled=False),
    ):
        logits = model(tokens).logits

        target_position = prefix_len - 1

        target_token = int(
            logits[
                0,
                target_position,
            ]
            .detach()
            .argmax()
            .item()
        )

        selected_logit = logits[
            0,
            target_position,
            target_token,
        ]

        selected_logit.backward()

    embedding_grad = captured["output"].grad

    past_grad_max = float(
        embedding_grad[
            :,
            :prefix_len,
        ]
        .abs()
        .max()
        .item()
    )

    future_grad_max = float(
        embedding_grad[
            :,
            prefix_len:,
        ]
        .abs()
        .max()
        .item()
    )

    hook.remove()

    passed = (
        past_grad_max > 0.0
        and future_grad_max <= 1e-12
    )

    result = {
        "arm": arm,
        "forward_numeric_delta": forward_max_abs,
        "past_grad_max": past_grad_max,
        "future_grad_max": future_grad_max,
        "pass": passed,
    }

    del (
        model,
        tokens,
        changed,
        logits_a,
        logits_b,
        logits,
        embedding_grad,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return result


causal_results = [
    causal_test("R3"),
    causal_test("M3"),
]

causal_df = pd.DataFrame(causal_results)

display(causal_df)

assert causal_df["pass"].all(), (
    "actual causal gradient sizintisi found; "
    "benchmarks do-not-run."
)

causal_df.to_csv(
    REPORT_ROOT / "causal_tests.csv",
    index=False,
)

print("Causal testlerin tamami passed.")

## 8. Discover shards and construct fixed validation batches

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 8 — Shard discovery and fixed corpus-validation batches
def collect_paths(obj) -> list[str]:
    found = []
    if isinstance(obj, str) and obj.endswith((".bin", ".npy")):
        found.append(obj)
    elif isinstance(obj, dict):
        for value in obj.values():
            found.extend(collect_paths(value))
    elif isinstance(obj, list):
        for value in obj:
            found.extend(collect_paths(value))
    return found


def resolve_shards(manifest_path: Path) -> list[Path]:
    manifest = json.loads(manifest_path.read_text())
    paths = []
    for item in collect_paths(manifest):
        p = Path(item)
        candidates = [p, manifest_path.parent / p, manifest_path.parent / p.name]
        hit = next((c.resolve() for c in candidates if c.exists()), None)
        if hit is not None and hit not in paths:
            paths.append(hit)
    if not paths:
        paths = sorted(manifest_path.parent.glob("*.bin"))
    return paths


def fixed_validation_batches(
    shard: Path,
    seq_len: int,
    batch_size: int,
    n_batches: int,
) -> list[torch.Tensor]:
    arr = (
        np.load(shard, mmap_mode="r")
        if shard.suffix == ".npy"
        else np.memmap(shard, mode="r", dtype=np.uint16)
    )
    required = batch_size * n_batches
    max_start = len(arr) - seq_len - 1
    if max_start <= 0:
        raise RuntimeError(f"Validation shard too short: {shard}")
    starts = np.linspace(0, max_start, required, dtype=np.int64)
    batches = []
    for batch_id in range(n_batches):
        selected = starts[batch_id * batch_size:(batch_id + 1) * batch_size]
        rows = [
            np.asarray(arr[s:s + seq_len], dtype=np.int64).copy()
            for s in selected
        ]
        batches.append(torch.from_numpy(np.stack(rows)))
    return batches


SHARDS = resolve_shards(MANIFEST_PATH) if MANIFEST_PATH is not None else []
VAL_SHARD = SHARDS[-1] if len(SHARDS) >= 2 else None
CORPUS_VAL_BATCHES = (
    fixed_validation_batches(
        VAL_SHARD,
        seq_len=512,
        batch_size=VAL_BATCH_SIZE,
        n_batches=VAL_BATCHES,
    )
    if VAL_SHARD is not None else []
)

print("found shard:", len(SHARDS))
print("Validation shard:", VAL_SHARD)
print("Validation batch:", len(CORPUS_VAL_BATCHES))


## 9. Define exact NLL, perplexity, and bootstrap utilities

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 9 — exact token-weighted NLL, perplexity and paired bootstrap tools
@torch.inference_mode()
def evaluate_logits_fn(
    model: nn.Module,
    batches: list[torch.Tensor],
    logits_fn: Callable[[nn.Module, torch.Tensor], torch.Tensor],
) -> tuple[dict, pd.DataFrame]:
    model.eval()
    sequence_rows = []
    torch.cuda.synchronize()
    start_time = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    for batch_id, cpu_batch in enumerate(batches):
        input_ids = cpu_batch.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = logits_fn(model, input_ids)
        targets = input_ids[:, 1:]
        token_nll = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction="none",
        ).view(input_ids.size(0), -1)
        seq_sum = token_nll.sum(dim=1).float().cpu().numpy()
        seq_count = np.full(input_ids.size(0), targets.size(1), dtype=np.int64)
        for row_id, (nll_sum, count) in enumerate(zip(seq_sum, seq_count)):
            sequence_rows.append({
                "batch_id": batch_id,
                "row_id": row_id,
                "nll_sum": float(nll_sum),
                "tokens": int(count),
                "mean_nll": float(nll_sum / count),
            })
        del input_ids, logits, targets, token_nll

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time
    frame = pd.DataFrame(sequence_rows)
    total_nll = frame["nll_sum"].sum()
    total_tokens = int(frame["tokens"].sum())
    loss = float(total_nll / total_tokens)
    result = {
        "loss": loss,
        "perplexity": float(math.exp(min(loss, 20.0))),
        "tokens": total_tokens,
        "seconds": elapsed,
        "tokens_per_second": total_tokens / elapsed,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
    }
    return result, frame


def normal_logits(model: nn.Module, input_ids: torch.Tensor) -> torch.Tensor:
    return model(input_ids).logits


def paired_bootstrap_loss_difference(
    frame_a: pd.DataFrame,
    frame_b: pd.DataFrame,
    n_boot: int = 5_000,
    seed: int = SEED,
) -> dict:
    a = frame_a.sort_values(["batch_id", "row_id"])["mean_nll"].to_numpy()
    b = frame_b.sort_values(["batch_id", "row_id"])["mean_nll"].to_numpy()
    assert len(a) == len(b)
    rng = np.random.default_rng(seed)
    differences = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        idx = rng.integers(0, len(a), size=len(a))
        differences[i] = np.mean(a[idx] - b[idx])
    return {
        "mean_a_minus_b": float(np.mean(a - b)),
        "ci95_low": float(np.quantile(differences, 0.025)),
        "ci95_high": float(np.quantile(differences, 0.975)),
        "prob_a_better": float(np.mean(differences < 0)),
    }


## 10. Compare R3 and M3 on the fixed corpus validation set

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 10 — Main corpus-validation comparison
BASE_VALIDATION = {}
BASE_SEQUENCE_FRAMES = {}

if not CORPUS_VAL_BATCHES:
    print("Corpus validation skipped: manifest/shard existing not.")
else:
    for arm in ("R3", "M3"):
        model, checkpoint = load_model(arm)
        result, frame = evaluate_logits_fn(model, CORPUS_VAL_BATCHES, normal_logits)
        result.update({
            "arm": arm,
            "parameters": model.num_trainable_parameters(),
            "seen_tokens": int(checkpoint["seen_tokens"]),
        })
        BASE_VALIDATION[arm] = result
        BASE_SEQUENCE_FRAMES[arm] = frame
        frame.assign(arm=arm).to_csv(
            REPORT_ROOT / f"{arm.lower()}_validation_per_sequence.csv",
            index=False,
        )
        print(json.dumps(result, indent=2))
        del model, checkpoint
        gc.collect()
        torch.cuda.empty_cache()

    base_df = pd.DataFrame(BASE_VALIDATION.values()).sort_values("loss")
    display(base_df)
    base_df.to_csv(REPORT_ROOT / "base_validation_summary.csv", index=False)

    paired = paired_bootstrap_loss_difference(
        BASE_SEQUENCE_FRAMES["R3"],
        BASE_SEQUENCE_FRAMES["M3"],
    )
    print("R3 loss - M3 loss paired bootstrap:")
    print(json.dumps(paired, indent=2))
    (REPORT_ROOT / "base_validation_paired_bootstrap.json").write_text(
        json.dumps(paired, indent=2), encoding="utf-8"
    )


## 11. Extract training curves from checkpoint history

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 11 — Extract training curves from checkpoint history
histories = []
for arm, path in CHECKPOINT_PATHS.items():
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    frame = pd.DataFrame(checkpoint.get("history", []))
    if not frame.empty:
        frame["arm"] = arm
        histories.append(frame)
    del checkpoint

if histories:
    history_df = pd.concat(histories, ignore_index=True)
    history_df.to_csv(REPORT_ROOT / "training_history.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    plot_specs = [
        ("loss", "Training loss"),
        ("lr", "Learning rate"),
        ("grad_norm", "Gradient norm"),
        ("tokens_per_second", "Training throughput"),
    ]
    for ax, (column, title) in zip(axes.flat, plot_specs):
        for arm, group in history_df.groupby("arm"):
            ordered = group.sort_values("tokens")
            smooth = ordered[column].rolling(25, min_periods=1).mean()
            ax.plot(ordered["tokens"] / 1e6, smooth, label=arm)
        ax.set_title(title)
        ax.set_xlabel("Seen tokens (million)")
        ax.grid(alpha=0.25)
        ax.legend()
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "training_curves.png", dpi=170)
    plt.show()
else:
    history_df = pd.DataFrame()
    print("Checkpoint history record not found.")


## 12. Define R3 inference ablation paths

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 12 — R3 ablation forward paths
@torch.inference_mode()
def r3_logits_mode(
    model: ANKAReflectiveLoop,
    input_ids: torch.Tensor,
    mode: str,
) -> torch.Tensor:
    token_state = model.embedding(input_ids)
    if token_state.is_cuda and cuda_autocast_enabled():
        token_state = token_state.to(torch.bfloat16)

    h = token_state + model.round_embedding[0].to(token_state.dtype)
    for block in model.blocks:
        h = block(h, None, None, model.residual_scale)
    draft = h

    if mode == "draft_only":
        return F.linear(model.draft_norm(draft), model.embedding.weight)

    discrepancy = model.reflect_delta_norm(draft - token_state)
    reflection = torch.tanh(model.reflect_proj(torch.cat([
        model.reflect_in_norm(draft), discrepancy
    ], dim=-1)))
    reflection = model.reflect_norm(reflection)

    learned_gates = torch.sigmoid(model.reflect_gate_logits)
    if mode == "learned":
        gates = learned_gates
    elif mode == "gate_zero":
        gates = torch.zeros_like(learned_gates)
    elif mode == "gate_one":
        gates = torch.ones_like(learned_gates)
    elif mode == "gate_mean":
        gates = torch.ones_like(learned_gates) * learned_gates.mean()
    elif mode == "shuffled_reflection":
        gates = learned_gates
        reflection = torch.roll(reflection, shifts=1, dims=0)
    else:
        raise KeyError(f"Bilinmeyen R3 mode: {mode}")

    h = draft + model.round_embedding[1].to(draft.dtype)
    for layer_id, block in enumerate(model.blocks):
        h = block(
            h,
            reflection,
            gates[layer_id].to(h.dtype),
            model.residual_scale,
        )
    return F.linear(model.final_norm(h), model.embedding.weight)


R3_ABLATION_MODES = [
    "learned",
    "draft_only",
    "gate_zero",
    "gate_one",
    "gate_mean",
    "shuffled_reflection",
]


## 13. Run R3 ablations and profile Reflect gates

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 13 — R3 ablationlarini run and gate profilini ciz
R3_ABLATIONS = []
R3_ABLATION_FRAMES = {}

if not CORPUS_VAL_BATCHES:
    print("R3 loss ablations skipped: corpus validation unavailable.")
else:
    r3_model, _ = load_model("R3")
    selected_batches = CORPUS_VAL_BATCHES[:ABLATION_BATCHES]

    # Verify that the custom learned path is equivalent to the original forward pass.
    probe = selected_batches[0][:2].to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        original = r3_model(probe).logits
        custom = r3_logits_mode(r3_model, probe, "learned")
    max_difference = float((original.float() - custom.float()).abs().max())
    print("R3 custom-forward max fark:", max_difference)
    assert max_difference < 0.05
    del probe, original, custom

    for mode in R3_ABLATION_MODES:
        result, frame = evaluate_logits_fn(
            r3_model,
            selected_batches,
            lambda model, ids, selected_mode=mode: r3_logits_mode(
                model, ids, selected_mode
            ),
        )
        result["mode"] = mode
        R3_ABLATIONS.append(result)
        R3_ABLATION_FRAMES[mode] = frame

    r3_ablation_df = pd.DataFrame(R3_ABLATIONS)
    learned_loss = float(
        r3_ablation_df.loc[r3_ablation_df["mode"] == "learned", "loss"].iloc[0]
    )
    r3_ablation_df["delta_loss_vs_learned"] = r3_ablation_df["loss"] - learned_loss
    display(r3_ablation_df.sort_values("loss"))
    r3_ablation_df.to_csv(REPORT_ROOT / "r3_ablation_summary.csv", index=False)

    gates = torch.sigmoid(r3_model.reflect_gate_logits).detach().cpu().numpy()
    gate_df = pd.DataFrame({
        "layer": np.arange(len(gates)),
        "gate": gates,
    })
    gate_df.to_csv(REPORT_ROOT / "r3_layer_gates.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    sns.barplot(data=r3_ablation_df, x="mode", y="delta_loss_vs_learned", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=35)
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("R3 ablation loss difference")
    axes[1].plot(gate_df["layer"], gate_df["gate"], marker="o")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("learned Reflect gate — layer by")
    axes[1].set_xlabel("Physical layer")
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "r3_ablation_and_gates.png", dpi=170)
    plt.show()

    del r3_model
    gc.collect()
    torch.cuda.empty_cache()


## 14. Measure token-level gains from the R3 second pass

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 14 — Measure where the R3 second pass improves token NLL
R3_POSITION_ANALYSIS = pd.DataFrame()
if CORPUS_VAL_BATCHES:
    r3_model, _ = load_model("R3")
    learned_sums = np.zeros(511, dtype=np.float64)
    draft_sums = np.zeros(511, dtype=np.float64)
    counts = np.zeros(511, dtype=np.int64)

    for cpu_batch in CORPUS_VAL_BATCHES[:ABLATION_BATCHES]:
        ids = cpu_batch.to(DEVICE, non_blocking=True)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            learned_logits = r3_logits_mode(r3_model, ids, "learned")
            draft_logits = r3_logits_mode(r3_model, ids, "draft_only")
        targets = ids[:, 1:]
        learned_nll = F.cross_entropy(
            learned_logits[:, :-1].reshape(-1, learned_logits.size(-1)),
            targets.reshape(-1), reduction="none"
        ).view(ids.size(0), -1)
        draft_nll = F.cross_entropy(
            draft_logits[:, :-1].reshape(-1, draft_logits.size(-1)),
            targets.reshape(-1), reduction="none"
        ).view(ids.size(0), -1)
        width = targets.size(1)
        learned_sums[:width] += learned_nll.sum(0).float().cpu().numpy()
        draft_sums[:width] += draft_nll.sum(0).float().cpu().numpy()
        counts[:width] += ids.size(0)
        del ids, learned_logits, draft_logits, targets, learned_nll, draft_nll

    valid = counts > 0
    R3_POSITION_ANALYSIS = pd.DataFrame({
        "position": np.arange(511)[valid],
        "learned_nll": learned_sums[valid] / counts[valid],
        "draft_nll": draft_sums[valid] / counts[valid],
    })
    R3_POSITION_ANALYSIS["reflection_gain"] = (
        R3_POSITION_ANALYSIS["draft_nll"] - R3_POSITION_ANALYSIS["learned_nll"]
    )
    R3_POSITION_ANALYSIS.to_csv(
        REPORT_ROOT / "r3_reflection_gain_by_position.csv", index=False
    )

    plt.figure(figsize=(13, 4))
    smooth = R3_POSITION_ANALYSIS["reflection_gain"].rolling(21, center=True).mean()
    plt.plot(R3_POSITION_ANALYSIS["position"], smooth)
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Second-pass NLL gain by position — higher is better")
    plt.xlabel("Token position")
    plt.ylabel("Draft NLL - final NLL")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / "r3_reflection_gain_by_position.png", dpi=170)
    plt.show()

    del r3_model
    gc.collect()
    torch.cuda.empty_cache()


## 15. Define alternative M3 routing paths

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 15 — M3 alternative routing paths
def deterministic_choices_from_load(
    count: int,
    load: torch.Tensor,
    layer_id: int,
) -> torch.Tensor:
    # same giriste each runmada same pseudo-random selection.
    index = torch.arange(count, device=load.device, dtype=torch.int64)
    raw = (index * 1_103_515_245 + (layer_id + 1) * 12_345) % 1_000_003
    u = raw.float() / 1_000_003.0
    cdf = load.float().cumsum(0)
    return (u[:, None] > cdf[None, :]).sum(dim=1).clamp_max(len(load) - 1)


def morph_dispatch_mode(
    x: torch.Tensor,
    morph: SparseMorphFFN,
    experts: nn.ModuleList,
    mode: str,
    layer_id: int,
) -> tuple[torch.Tensor, dict]:
    shape = x.shape
    flat = x.reshape(-1, shape[-1])
    probs = torch.softmax(morph.router(flat).float(), dim=-1)
    learned_choice = probs.argmax(dim=-1)

    if mode == "learned":
        choice = learned_choice
    elif mode == "random_load_matched":
        load = F.one_hot(learned_choice, morph.n_experts).float().mean(0)
        choice = deterministic_choices_from_load(len(flat), load, layer_id)
    elif mode == "uniform_hash":
        choice = (torch.arange(len(flat), device=flat.device) + layer_id) % morph.n_experts
    elif mode.startswith("expert_"):
        expert_id = int(mode.split("_")[-1])
        choice = torch.full_like(learned_choice, expert_id)
    elif mode == "soft_mixture":
        out = torch.zeros_like(flat)
        for expert_id, expert in enumerate(experts):
            expert_out = expert(flat)
            out = out + expert_out * probs[:, expert_id:expert_id + 1].to(expert_out.dtype)
        stats = {
            "load": probs.mean(0).detach(),
            "entropy": (-(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()).detach(),
            "confidence": probs.max(-1).values.mean().detach(),
        }
        return out.view(shape), stats
    else:
        raise KeyError(f"Bilinmeyen M3 routing mode: {mode}")

    out = torch.zeros_like(flat)
    for expert_id, expert in enumerate(experts):
        indices = torch.nonzero(choice == expert_id, as_tuple=False).flatten()
        if indices.numel() == 0:
            continue
        expert_out = expert(flat.index_select(0, indices))
        out = out.index_copy(0, indices, expert_out.to(out.dtype))

    stats = {
        "load": F.one_hot(choice, morph.n_experts).float().mean(0).detach(),
        "entropy": (-(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()).detach(),
        "confidence": probs.max(-1).values.mean().detach(),
    }
    return out.view(shape), stats


@torch.inference_mode()
def m3_logits_mode(
    model: ANKASparseMorph,
    input_ids: torch.Tensor,
    mode: str,
    return_stats: bool = False,
):
    h = model.embedding(input_ids)
    if h.is_cuda and cuda_autocast_enabled():
        h = h.to(torch.bfloat16)
    all_stats = []
    for layer_id, block in enumerate(model.blocks):
        stage_id = layer_id // model.layers_per_stage
        h = h + block.residual_scale * block.attn(block.attn_norm(h))
        routed, stats = morph_dispatch_mode(
            block.ffn_norm(h),
            block.morph,
            model.stage_experts[stage_id],
            mode,
            layer_id,
        )
        h = h + block.residual_scale * routed
        all_stats.append(stats)
    logits = F.linear(model.final_norm(h), model.embedding.weight)
    return (logits, all_stats) if return_stats else logits


M3_ROUTING_MODES = [
    "learned",
    "random_load_matched",
    "uniform_hash",
    "expert_0",
    "expert_1",
    "expert_2",
    "expert_3",
]


## 16. Run M3 routing ablations with matched forward paths

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 16 — Run M3 routing ablations with correct forward paths

# this cell:
# 1. original M3 forward'ini exactly again produces.
# 2. exactly equivalence validates.
# 3. Routing testlerinde straight-through numerical effect ayirir.
# 4. Compare the soft-mixture result on the same validation subset.


def m3_deterministic_choices(
    count: int,
    load: torch.Tensor,
    layer_id: int,
) -> torch.Tensor:
    """Deterministic pseudo-random expert selection matched to the load distribution."""

    indices = torch.arange(
        count,
        device=load.device,
        dtype=torch.int64,
    )

    raw = (
        indices * 1_103_515_245
        + (layer_id + 1) * 12_345
    ) % 1_000_003

    uniform_values = raw.float() / 1_000_003.0
    cumulative_load = load.float().cumsum(dim=0)

    choices = (
        uniform_values[:, None]
        > cumulative_load[None, :]
    ).sum(dim=1)

    return choices.clamp_max(load.numel() - 1)


def m3_dispatch(
    x: torch.Tensor,
    morph: SparseMorphFFN,
    experts: nn.ModuleList,
    mode: str,
    layer_id: int,
    reproduce_training_forward: bool = False,
) -> torch.Tensor:
    """
    M3 expert dispatch.

    reproduce_training_forward=True:
        Exactly reproduces the training-time/original forward path.

    reproduce_training_forward=False:
        Straight-through carpanini inference during kaldirir.
        This prevents routing ablations from being confounded by numerical BF16 effects.
    """

    original_shape = x.shape
    flat = x.reshape(-1, original_shape[-1])

    router_logits = morph.router(flat).float()
    probabilities = torch.softmax(
        router_logits,
        dim=-1,
    )

    learned_choices = probabilities.argmax(dim=-1)

    if mode == "learned":
        choices = learned_choices

    elif mode == "random_load_matched":
        learned_load = F.one_hot(
            learned_choices,
            num_classes=morph.n_experts,
        ).float().mean(dim=0)

        choices = m3_deterministic_choices(
            count=flat.size(0),
            load=learned_load,
            layer_id=layer_id,
        )

    elif mode == "uniform_hash":
        choices = (
            torch.arange(
                flat.size(0),
                device=flat.device,
            )
            + layer_id
        ) % morph.n_experts

    elif mode.startswith("expert_"):
        expert_id = int(mode.split("_")[-1])

        if not 0 <= expert_id < morph.n_experts:
            raise ValueError(
                f"Gecersiz expert numarasi: {expert_id}"
            )

        choices = torch.full_like(
            learned_choices,
            expert_id,
        )

    elif mode == "soft_mixture":
        output = torch.zeros_like(flat)

        for expert_id, expert in enumerate(experts):
            expert_output = expert(flat)

            expert_probability = probabilities[
                :,
                expert_id:expert_id + 1,
            ].to(expert_output.dtype)

            output = (
                output
                + expert_output * expert_probability
            )

        return output.view(original_shape)

    else:
        raise KeyError(
            f"Bilinmeyen M3 routing modu: {mode}"
        )

    output = torch.zeros_like(flat)

    for expert_id, expert in enumerate(experts):
        selected_indices = torch.nonzero(
            choices == expert_id,
            as_tuple=False,
        ).flatten()

        if selected_indices.numel() == 0:
            continue

        selected_input = flat.index_select(
            0,
            selected_indices,
        )

        expert_output = expert(selected_input)

        if reproduce_training_forward:
            # original SparseMorphFFN.forward with exactly same path.
            selected_probability = probabilities.index_select(
                0,
                selected_indices,
            )[:, expert_id].to(expert_output.dtype)

            straight_through = (
                1.0
                + selected_probability
                - selected_probability.detach()
            )

            expert_output = (
                expert_output
                * straight_through.unsqueeze(-1)
            )

        output = output.index_copy(
            0,
            selected_indices,
            expert_output.to(output.dtype),
        )

    return output.view(original_shape)


@torch.inference_mode()
def m3_logits_ablation(
    model: ANKASparseMorph,
    input_ids: torch.Tensor,
    mode: str,
    reproduce_training_forward: bool = False,
) -> torch.Tensor:
    """Runs the M3 forward path with the selected routing method."""

    hidden = model.embedding(input_ids)

    if hidden.is_cuda and cuda_autocast_enabled():
        hidden = hidden.to(torch.bfloat16)

    for layer_id, block in enumerate(model.blocks):
        stage_id = (
            layer_id
            // model.layers_per_stage
        )

        attention_input = block.attn_norm(hidden)

        attention_output = block.attn(
            attention_input
        )

        hidden = (
            hidden
            + block.residual_scale
            * attention_output
        )

        morph_input = block.ffn_norm(hidden)

        morph_output = m3_dispatch(
            x=morph_input,
            morph=block.morph,
            experts=model.stage_experts[stage_id],
            mode=mode,
            layer_id=layer_id,
            reproduce_training_forward=(
                reproduce_training_forward
            ),
        )

        hidden = (
            hidden
            + block.residual_scale
            * morph_output
        )

    logits = F.linear(
        model.final_norm(hidden),
        model.embedding.weight,
    )

    return logits


M3_ROUTING_MODES = [
    "learned",
    "random_load_matched",
    "uniform_hash",
    "expert_0",
    "expert_1",
    "expert_2",
    "expert_3",
]


M3_ABLATIONS = []
M3_ABLATION_FRAMES = {}


if not CORPUS_VAL_BATCHES:
    print(
        "M3 loss ablations skipped: "
        "corpus validation batches not found."
    )

else:
    m3_model, _ = load_model("M3")
    m3_model.eval()

    selected_batches = CORPUS_VAL_BATCHES[
        :ABLATION_BATCHES
    ]

    # =========================================================
    # 1. CUSTOM FORWARD ESDEGERLIK TESTI
    # =========================================================
    probe = selected_batches[0][:2].to(
        DEVICE,
        non_blocking=True,
    )

    with (
        torch.inference_mode(),
        torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ),
    ):
        original_logits = m3_model(
            probe
        ).logits

        reproduced_logits = m3_logits_ablation(
            model=m3_model,
            input_ids=probe,
            mode="learned",
            reproduce_training_forward=True,
        )

    max_difference = float(
        (
            original_logits.float()
            - reproduced_logits.float()
        )
        .abs()
        .max()
        .item()
    )

    mean_difference = float(
        (
            original_logits.float()
            - reproduced_logits.float()
        )
        .abs()
        .mean()
        .item()
    )

    print(
        "M3 exact custom-forward max fark :",
        max_difference,
    )

    print(
        "M3 exact custom-forward ort fark :",
        mean_difference,
    )

    assert max_difference < 1e-3, (
        "The custom M3 forward still differs from the original forward by "
        f"does-not-match. Max fark: {max_difference}"
    )

    del (
        probe,
        original_logits,
        reproduced_logits,
    )

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "M3 custom-forward esdegerlik testi: PASS"
    )

    # =========================================================
    # 2. ORIGINAL CHECKPOINT FORWARD RESULT
    # =========================================================
    original_result, original_frame = (
        evaluate_logits_fn(
            m3_model,
            selected_batches,
            normal_logits,
        )
    )

    original_result.update({
        "mode": "original_forward",
        "evaluation_scope": "full",
    })

    M3_ABLATIONS.append(
        original_result
    )

    M3_ABLATION_FRAMES[
        "original_forward"
    ] = original_frame

    # =========================================================
    # 3. clean ROUTING ablations
    # =========================================================
    # Inference during straight-through carpani kaldirilir.
    # This isolates the effect of expert selection.
    for routing_mode in M3_ROUTING_MODES:
        result, frame = evaluate_logits_fn(
            m3_model,
            selected_batches,
            lambda model, ids, selected_mode=routing_mode: (
                m3_logits_ablation(
                    model=model,
                    input_ids=ids,
                    mode=selected_mode,
                    reproduce_training_forward=False,
                )
            ),
        )

        result.update({
            "mode": routing_mode,
            "evaluation_scope": "full",
        })

        M3_ABLATIONS.append(result)
        M3_ABLATION_FRAMES[
            routing_mode
        ] = frame

    # =========================================================
    # 4. SOFT MIXTURE
    # =========================================================
    # Use a smaller subset because all four experts are evaluated.
    soft_batch_count = max(
        2,
        min(
            5,
            len(selected_batches),
        ),
    )

    soft_batches = selected_batches[
        :soft_batch_count
    ]

    soft_result, soft_frame = evaluate_logits_fn(
        m3_model,
        soft_batches,
        lambda model, ids: m3_logits_ablation(
            model=model,
            input_ids=ids,
            mode="soft_mixture",
            reproduce_training_forward=False,
        ),
    )

    learned_subset_result, _ = evaluate_logits_fn(
        m3_model,
        soft_batches,
        lambda model, ids: m3_logits_ablation(
            model=model,
            input_ids=ids,
            mode="learned",
            reproduce_training_forward=False,
        ),
    )

    soft_result.update({
        "mode": "soft_mixture",
        "evaluation_scope": (
            f"first_{soft_batch_count}_batches"
        ),
        "subset_learned_loss": (
            learned_subset_result["loss"]
        ),
        "subset_delta_loss": (
            soft_result["loss"]
            - learned_subset_result["loss"]
        ),
    })

    M3_ABLATIONS.append(
        soft_result
    )

    M3_ABLATION_FRAMES[
        "soft_mixture"
    ] = soft_frame

    # =========================================================
    # 5. RESULT TABLOSU
    # =========================================================
    m3_ablation_df = pd.DataFrame(
        M3_ABLATIONS
    )

    learned_clean_loss = float(
        m3_ablation_df.loc[
            m3_ablation_df["mode"] == "learned",
            "loss",
        ].iloc[0]
    )

    m3_ablation_df[
        "delta_loss_vs_learned"
    ] = (
        m3_ablation_df["loss"]
        - learned_clean_loss
    )

    # Soft mixture only kendi alt in-subset
    # Compare against the learned-routing result.
    soft_mask = (
        m3_ablation_df["mode"]
        == "soft_mixture"
    )

    m3_ablation_df.loc[
        soft_mask,
        "delta_loss_vs_learned",
    ] = m3_ablation_df.loc[
        soft_mask,
        "subset_delta_loss",
    ]

    m3_ablation_df = (
        m3_ablation_df
        .sort_values("loss")
        .reset_index(drop=True)
    )

    display(
        m3_ablation_df[
            [
                "mode",
                "evaluation_scope",
                "loss",
                "perplexity",
                "delta_loss_vs_learned",
                "tokens_per_second",
                "peak_vram_gib",
            ]
        ]
    )

    m3_ablation_df.to_csv(
        REPORT_ROOT
        / "m3_ablation_summary.csv",
        index=False,
    )

    # =========================================================
    # 6. GRAFIK
    # =========================================================
    plt.figure(
        figsize=(14, 5)
    )

    sns.barplot(
        data=m3_ablation_df,
        x="mode",
        y="delta_loss_vs_learned",
    )

    plt.xticks(
        rotation=35,
        ha="right",
    )

    plt.axhline(
        0,
        color="black",
        linewidth=1,
    )

    plt.title(
        "M3 routing ablation loss difference\n"
        "Negative values indicate improvement over learned routing"
    )

    plt.ylabel(
        "Loss difference"
    )

    plt.xlabel(
        "Routing modu"
    )

    plt.tight_layout()

    plt.savefig(
        REPORT_ROOT
        / "m3_routing_ablation.png",
        dpi=170,
    )

    plt.show()

    del m3_model

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "M3 routing ablations completed."
    )

## 17. Measure router load, entropy, confidence, and specialization

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 17 — M3 router yuku, entropy, confidence and token–expert relationship
ROUTER_DIAGNOSTICS = pd.DataFrame()
ROUTER_TOP_TOKENS = []

if CORPUS_VAL_BATCHES:
    m3_model, _ = load_model("M3")
    layer_records = defaultdict(lambda: {
        "choices": [], "tokens": [], "confidence": [], "entropy": []
    })
    handles = []

    def make_router_hook(layer_id: int):
        def hook(module, inputs):
            x = inputs[0]
            flat = x.detach().reshape(-1, x.size(-1))
            probs = torch.softmax(module.router(flat).float(), dim=-1)
            choices = probs.argmax(-1)
            record = layer_records[layer_id]
            record["choices"].append(choices.cpu())
            record["confidence"].append(probs.max(-1).values.cpu())
            record["entropy"].append(
                (-(probs * probs.clamp_min(1e-9).log()).sum(-1)).cpu()
            )
        return hook

    for layer_id, block in enumerate(m3_model.blocks):
        handles.append(block.morph.register_forward_pre_hook(make_router_hook(layer_id)))

    diagnostic_batches = CORPUS_VAL_BATCHES[:min(10, len(CORPUS_VAL_BATCHES))]
    with torch.inference_mode():
        for cpu_batch in diagnostic_batches:
            ids = cpu_batch.to(DEVICE, non_blocking=True)
            for layer_id in range(len(m3_model.blocks)):
                layer_records[layer_id]["tokens"].append(ids.reshape(-1).cpu())
            with torch.autocast("cuda", dtype=torch.bfloat16):
                _ = m3_model(ids)

    for handle in handles:
        handle.remove()

    diagnostic_rows = []
    for layer_id in range(len(m3_model.blocks)):
        choices = torch.cat(layer_records[layer_id]["choices"]).numpy()
        tokens = torch.cat(layer_records[layer_id]["tokens"]).numpy()
        confidence = torch.cat(layer_records[layer_id]["confidence"]).numpy()
        entropy = torch.cat(layer_records[layer_id]["entropy"]).numpy()
        stage_id = layer_id // m3_model.layers_per_stage
        load = np.bincount(choices, minlength=4) / len(choices)
        nmi = normalized_mutual_info_score(tokens, choices)
        row = {
            "layer": layer_id,
            "stage": stage_id,
            "normalized_entropy": float(entropy.mean() / math.log(4)),
            "mean_confidence": float(confidence.mean()),
            "token_expert_nmi": float(nmi),
        }
        for expert_id, value in enumerate(load):
            row[f"expert_{expert_id}_load"] = float(value)
        diagnostic_rows.append(row)

        for expert_id in range(4):
            counts = Counter(tokens[choices == expert_id].tolist())
            for rank, (token_id, count) in enumerate(counts.most_common(15), 1):
                ROUTER_TOP_TOKENS.append({
                    "layer": layer_id,
                    "stage": stage_id,
                    "expert": expert_id,
                    "rank": rank,
                    "token_id": int(token_id),
                    "token": TOKENIZER.decode([int(token_id)]),
                    "count": int(count),
                })

    ROUTER_DIAGNOSTICS = pd.DataFrame(diagnostic_rows)
    top_tokens_df = pd.DataFrame(ROUTER_TOP_TOKENS)
    display(ROUTER_DIAGNOSTICS)
    display(top_tokens_df.head(30))
    ROUTER_DIAGNOSTICS.to_csv(REPORT_ROOT / "m3_router_diagnostics.csv", index=False)
    top_tokens_df.to_csv(REPORT_ROOT / "m3_router_top_tokens.csv", index=False)

    load_columns = [f"expert_{i}_load" for i in range(4)]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(
        ROUTER_DIAGNOSTICS.set_index("layer")[load_columns],
        annot=True, fmt=".2f", cmap="viridis", ax=axes[0]
    )
    axes[0].set_title("M3 layer × expert yuku")
    axes[1].plot(
        ROUTER_DIAGNOSTICS["layer"],
        ROUTER_DIAGNOSTICS["normalized_entropy"],
        marker="o", label="Normalized entropy"
    )
    axes[1].plot(
        ROUTER_DIAGNOSTICS["layer"],
        ROUTER_DIAGNOSTICS["mean_confidence"],
        marker="o", label="Top-1 confidence"
    )
    axes[1].plot(
        ROUTER_DIAGNOSTICS["layer"],
        ROUTER_DIAGNOSTICS["token_expert_nmi"],
        marker="o", label="Token–expert NMI"
    )
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Router behavior")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "m3_router_diagnostics.png", dpi=170)
    plt.show()

    del m3_model
    gc.collect()
    torch.cuda.empty_cache()


## 18. Measure expert-output similarity

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 18 — Measure expert-output similarity on matched inputs
EXPERT_SIMILARITY = pd.DataFrame()
if CORPUS_VAL_BATCHES:
    m3_model, _ = load_model("M3")
    captured = {}
    handles = []

    for stage_id in range(m3_model.n_stages):
        layer_id = stage_id * m3_model.layers_per_stage

        def make_capture(selected_stage: int):
            def hook(module, inputs):
                if selected_stage not in captured:
                    captured[selected_stage] = inputs[0].detach()[:1, :128].clone()
            return hook

        handles.append(
            m3_model.blocks[layer_id].morph.register_forward_pre_hook(
                make_capture(stage_id)
            )
        )

    ids = CORPUS_VAL_BATCHES[0].to(DEVICE)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        _ = m3_model(ids)
    for handle in handles:
        handle.remove()

    rows = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        for stage_id, x in captured.items():
            flat = x.reshape(-1, x.size(-1))
            outputs = [expert(flat).float() for expert in m3_model.stage_experts[stage_id]]
            for i in range(4):
                for j in range(i + 1, 4):
                    similarity = F.cosine_similarity(outputs[i], outputs[j], dim=-1).mean()
                    rows.append({
                        "stage": stage_id,
                        "expert_a": i,
                        "expert_b": j,
                        "mean_cosine_similarity": float(similarity),
                    })

    EXPERT_SIMILARITY = pd.DataFrame(rows)
    display(EXPERT_SIMILARITY.groupby("stage")["mean_cosine_similarity"].agg(["mean", "min", "max"]))
    EXPERT_SIMILARITY.to_csv(REPORT_ROOT / "m3_expert_similarity.csv", index=False)

    del m3_model, ids
    gc.collect()
    torch.cuda.empty_cache()


## 19. Define the variable-length multiple-choice scoring engine

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 19 — Variable-length multiple-choice benchmark scoring engine
def encode_text(text: str) -> list[int]:
    return TOKENIZER.encode(text).ids


def prepare_candidate(
    context: str,
    continuation: str,
    max_length: int,
) -> tuple[list[int], int]:
    context_ids = encode_text(context)
    continuation_ids = encode_text(continuation)
    if not context_ids:
        context_ids = [BOS_ID]
    if not continuation_ids:
        continuation_ids = [EOS_ID] if EOS_ID is not None else [BOS_ID]
    available_context = max(1, max_length - len(continuation_ids))
    context_ids = context_ids[-available_context:]
    ids = (context_ids + continuation_ids)[-max_length:]
    continuation_start = len(ids) - len(continuation_ids)
    continuation_start = max(1, continuation_start)
    return ids, continuation_start


@torch.inference_mode()
def score_candidate_entries(
    model: nn.Module,
    entries: list[dict],
    batch_size: int = SCORE_BATCH_SIZE,
) -> pd.DataFrame:
    model.eval()
    prepared = []
    for entry in entries:
        ids, start = prepare_candidate(
            entry["context"], entry["continuation"], model.config.max_seq_len
        )
        prepared.append((entry, ids, start))

    rows = []
    for batch_start in range(0, len(prepared), batch_size):
        group = prepared[batch_start:batch_start + batch_size]
        width = max(len(ids) for _, ids, _ in group)
        input_ids = torch.full(
            (len(group), width), PAD_ID, dtype=torch.long, device=DEVICE
        )
        score_mask = torch.zeros(
            (len(group), width - 1), dtype=torch.bool, device=DEVICE
        )
        lengths = []
        for row_id, (_, ids, continuation_start) in enumerate(group):
            input_ids[row_id, :len(ids)] = torch.tensor(ids, device=DEVICE)
            # Target position j is predicted by logit position j-1.
            score_mask[row_id, continuation_start - 1:len(ids) - 1] = True
            lengths.append(len(ids))

        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids).logits
        targets = input_ids[:, 1:]
        token_nll = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction="none",
        ).view(len(group), -1)
        greedy = logits[:, :-1].argmax(-1)

        for row_id, ((entry, _, _), length) in enumerate(zip(group, lengths)):
            mask = score_mask[row_id]
            selected_nll = token_nll[row_id][mask]
            token_count = int(mask.sum())
            exact = bool(torch.all(greedy[row_id][mask] == targets[row_id][mask]))
            rows.append({
                **entry,
                "raw_loglikelihood": -float(selected_nll.sum()),
                "normalized_loglikelihood": -float(selected_nll.mean()),
                "continuation_tokens": token_count,
                "teacher_forced_exact": exact,
            })
        del input_ids, score_mask, logits, targets, token_nll, greedy
    return pd.DataFrame(rows)


def expand_benchmark_items(items: list[dict]) -> list[dict]:
    entries = []
    for item_index, item in enumerate(items):
        for choice_index, choice in enumerate(item["choices"]):
            entries.append({
                "item_index": item_index,
                "item_id": item.get("id", str(item_index)),
                "benchmark": item["benchmark"],
                "category": item.get("category", item["benchmark"]),
                "context": item["context"],
                "continuation": choice,
                "choice_index": choice_index,
                "label": int(item["label"]),
            })
    return entries


def predictions_from_candidate_scores(scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for item_index, group in scores.groupby("item_index", sort=True):
        raw_row = group.loc[group["raw_loglikelihood"].idxmax()]
        norm_row = group.loc[group["normalized_loglikelihood"].idxmax()]
        label = int(group["label"].iloc[0])
        rows.append({
            "item_index": int(item_index),
            "item_id": group["item_id"].iloc[0],
            "benchmark": group["benchmark"].iloc[0],
            "category": group["category"].iloc[0],
            "label": label,
            "raw_prediction": int(raw_row["choice_index"]),
            "normalized_prediction": int(norm_row["choice_index"]),
            "raw_correct": int(raw_row["choice_index"]) == label,
            "normalized_correct": int(norm_row["choice_index"]) == label,
        })
    return pd.DataFrame(rows)


def run_multiple_choice_items(arm: str, items: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    model, _ = load_model(arm)
    entries = expand_benchmark_items(items)
    scores = score_candidate_entries(model, entries)
    predictions = predictions_from_candidate_scores(scores)
    scores["arm"] = arm
    predictions["arm"] = arm
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return scores, predictions


## 20. Build the controlled English skills suite

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 20 — Build the controlled English skills suite
def shuffled_item(
    rng: random.Random,
    benchmark: str,
    category: str,
    context: str,
    correct: str,
    wrong: str,
    item_id: str,
) -> dict:
    choices = [correct, wrong]
    rng.shuffle(choices)
    return {
        "id": item_id,
        "benchmark": benchmark,
        "category": category,
        "context": context,
        "choices": choices,
        "label": choices.index(correct),
    }


def build_controlled_suite(per_task: int, seed: int = SEED) -> list[dict]:
    rng = random.Random(seed)
    names = ["Alice", "Bob", "Carol", "David", "Emma", "Frank", "Grace", "Henry"]
    colors = ["red", "blue", "green", "yellow", "black", "white"]
    objects = ["key", "coin", "book", "ball", "cup", "map"]
    rooms = ["kitchen", "garden", "office", "bedroom", "hall", "garage"]
    items = []

    for i in range(per_task):
        a, b, c = rng.sample(names, 3)
        color_a, color_b = rng.sample(colors, 2)
        obj = rng.choice(objects)
        room = rng.choice(rooms)

        items.append(shuffled_item(
            rng, "anka_controlled", "entity_tracking",
            f"{a} carries the {color_a} {obj}. {b} carries the {color_b} {obj}. "
            f"The person carrying the {color_a} {obj} is",
            f" {a}.", f" {b}.", f"entity_{i}"
        ))

        items.append(shuffled_item(
            rng, "anka_controlled", "transitive_order",
            f"{a} is older than {b}. {b} is older than {c}. Therefore, {a} is",
            f" older than {c}.", f" younger than {c}.", f"transitive_{i}"
        ))

        first, second, third = rng.sample([
            "the lamp turned on", "the door opened", "the bell rang",
            "the glass fell", "the clock stopped", "the dog barked"
        ], 3)
        items.append(shuffled_item(
            rng, "anka_controlled", "temporal_order",
            f"First, {first}. Later, {second}. Finally, {third}. The earliest event was",
            f" {first}.", f" {third}.", f"temporal_{i}"
        ))

        items.append(shuffled_item(
            rng, "anka_controlled", "negation",
            f"The {obj} is not {color_a}. It is {color_b}. The {obj} is",
            f" {color_b}.", f" {color_a}.", f"negation_{i}"
        ))

        items.append(shuffled_item(
            rng, "anka_controlled", "containment",
            f"The {obj} is inside a box. The box is inside the {room}. The {obj} is",
            f" inside the {room}.", f" outside the {room}.", f"containment_{i}"
        ))

        cause_templates = [
            ("dropped a glass onto a stone floor", " broke.", " floated away."),
            ("left ice under the hot sun", " melted.", " became colder."),
            ("forgot to water the plant for many weeks", " dried out.", " grew rapidly."),
            ("pushed a ball down a steep hill", " rolled downward.", " stayed in the air."),
        ]
        cause, correct, wrong = rng.choice(cause_templates)
        items.append(shuffled_item(
            rng, "anka_controlled", "physical_causality",
            f"{a} {cause}. The most likely result is that it",
            correct, wrong, f"causality_{i}"
        ))

        items.append(shuffled_item(
            rng, "anka_controlled", "coreference",
            f"{a} thanked {b} because {b} helped {a}. The person who provided help was",
            f" {b}.", f" {a}.", f"coreference_{i}"
        ))

    return items


CONTROLLED_ITEMS = build_controlled_suite(CONTROLLED_PER_TASK)
print("Controlled test items:", len(CONTROLLED_ITEMS))
print("Kategoriler:", sorted({item["category"] for item in CONTROLLED_ITEMS}))


## 21. Evaluate R3 and M3 on controlled skills

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 21 — Evaluate R3 and M3 on the controlled suite
CONTROLLED_SCORES = []
CONTROLLED_PREDICTIONS = []

for arm in ("R3", "M3"):
    scores, predictions = run_multiple_choice_items(arm, CONTROLLED_ITEMS)
    CONTROLLED_SCORES.append(scores)
    CONTROLLED_PREDICTIONS.append(predictions)

controlled_scores_df = pd.concat(CONTROLLED_SCORES, ignore_index=True)
controlled_predictions_df = pd.concat(CONTROLLED_PREDICTIONS, ignore_index=True)
controlled_summary_df = (
    controlled_predictions_df
    .groupby(["arm", "category"])[["raw_correct", "normalized_correct"]]
    .mean()
    .reset_index()
)

display(controlled_summary_df)
controlled_scores_df.to_csv(REPORT_ROOT / "controlled_candidate_scores.csv", index=False)
controlled_predictions_df.to_csv(REPORT_ROOT / "controlled_predictions.csv", index=False)
controlled_summary_df.to_csv(REPORT_ROOT / "controlled_summary.csv", index=False)

plt.figure(figsize=(13, 5))
sns.barplot(
    data=controlled_summary_df,
    x="category", y="normalized_correct", hue="arm"
)
plt.axhline(0.5, color="black", linestyle="--", label="Chance")
plt.xticks(rotation=35, ha="right")
plt.ylim(0, 1)
plt.title("ANKA controlled English skills suite")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "controlled_benchmark.png", dpi=170)
plt.show()


## 22. Load and normalize external zero-shot benchmarks

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 22 — Download external benchmarks and convert them to a shared format
from datasets import get_dataset_config_names, load_dataset


def take_limit(dataset, limit: int):
    count = min(limit, len(dataset))
    return dataset.select(range(count))


def load_external_benchmarks() -> tuple[list[dict], list[dict]]:
    items = []
    errors = []

    # HellaSwag
    try:
        ds = take_limit(load_dataset("Rowan/hellaswag", split="validation"), EXTERNAL_LIMIT)
        for i, row in enumerate(ds):
            label = int(row["label"])
            if label < 0:
                continue
            items.append({
                "id": f"hellaswag_{i}",
                "benchmark": "hellaswag",
                "category": "commonsense_completion",
                "context": row["ctx"],
                "choices": [" " + ending.lstrip() for ending in row["endings"]],
                "label": label,
            })
    except Exception as exc:
        errors.append({"benchmark": "hellaswag", "error": repr(exc)})

    # PIQA
    try:
        try:
            ds = load_dataset("ybisk/piqa", split="validation")
        except Exception:
            ds = load_dataset("piqa", split="validation", trust_remote_code=True)
        ds = take_limit(ds, EXTERNAL_LIMIT)
        for i, row in enumerate(ds):
            items.append({
                "id": f"piqa_{i}",
                "benchmark": "piqa",
                "category": "physical_commonsense",
                "context": f"Goal: {row['goal']}\nSolution:",
                "choices": [" " + row["sol1"].lstrip(), " " + row["sol2"].lstrip()],
                "label": int(row["label"]),
            })
    except Exception as exc:
        errors.append({"benchmark": "piqa", "error": repr(exc)})

    # WinoGrande
    try:
        ds = load_dataset(
            "allenai/winogrande", "winogrande_xl", split="validation"
        )
        ds = take_limit(ds, EXTERNAL_LIMIT)
        for i, row in enumerate(ds):
            answer = int(row["answer"]) - 1
            choices = [
                " " + row["sentence"].replace("_", row["option1"]),
                " " + row["sentence"].replace("_", row["option2"]),
            ]
            items.append({
                "id": f"winogrande_{i}",
                "benchmark": "winogrande",
                "category": "coreference",
                "context": "",
                "choices": choices,
                "label": answer,
            })
    except Exception as exc:
        errors.append({"benchmark": "winogrande", "error": repr(exc)})

    # ARC-Easy
    try:
        ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
        ds = take_limit(ds, EXTERNAL_LIMIT)
        for i, row in enumerate(ds):
            labels = [str(x) for x in row["choices"]["label"]]
            answer_key = str(row["answerKey"])
            if answer_key not in labels:
                continue
            items.append({
                "id": f"arc_easy_{i}",
                "benchmark": "arc_easy",
                "category": "science_qa",
                "context": f"Question: {row['question']}\nAnswer:",
                "choices": [" " + str(x).lstrip() for x in row["choices"]["text"]],
                "label": labels.index(answer_key),
            })
    except Exception as exc:
        errors.append({"benchmark": "arc_easy", "error": repr(exc)})

    # BLiMP — all configlerden fixed sayida minimal pair.
    try:
        configs = get_dataset_config_names("nyu-mll/blimp")
        for config_name in configs:
            ds = load_dataset("nyu-mll/blimp", config_name, split="train")
            ds = take_limit(ds, BLIMP_PER_CONFIG)
            for i, row in enumerate(ds):
                items.append({
                    "id": f"blimp_{config_name}_{i}",
                    "benchmark": "blimp",
                    "category": config_name,
                    "context": "",
                    "choices": [" " + row["sentence_good"], " " + row["sentence_bad"]],
                    "label": 0,
                })
    except Exception as exc:
        errors.append({"benchmark": "blimp", "error": repr(exc)})

    return items, errors


EXTERNAL_ITEMS, EXTERNAL_LOAD_ERRORS = load_external_benchmarks()
print("external benchmark maddesi:", len(EXTERNAL_ITEMS))
print(pd.Series([item["benchmark"] for item in EXTERNAL_ITEMS]).value_counts())
if EXTERNAL_LOAD_ERRORS:
    print("Benchmarks that could not be loaded:")
    display(pd.DataFrame(EXTERNAL_LOAD_ERRORS))
(REPORT_ROOT / "external_dataset_load_errors.json").write_text(
    json.dumps(EXTERNAL_LOAD_ERRORS, indent=2), encoding="utf-8"
)


## 23. Evaluate R3 and M3 on external benchmarks

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 23 — external benchmarks R3 and M3 on run
EXTERNAL_SCORES = []
EXTERNAL_PREDICTIONS = []

if not EXTERNAL_ITEMS:
    print("No external benchmark could be loaded; this section was skipped.")
    external_scores_df = pd.DataFrame()
    external_predictions_df = pd.DataFrame()
    external_summary_df = pd.DataFrame()
else:
    for arm in ("R3", "M3"):
        print(f"{arm} external benchmarks works...")
        scores, predictions = run_multiple_choice_items(arm, EXTERNAL_ITEMS)
        EXTERNAL_SCORES.append(scores)
        EXTERNAL_PREDICTIONS.append(predictions)

    external_scores_df = pd.concat(EXTERNAL_SCORES, ignore_index=True)
    external_predictions_df = pd.concat(EXTERNAL_PREDICTIONS, ignore_index=True)
    external_summary_df = (
        external_predictions_df
        .groupby(["arm", "benchmark"])[["raw_correct", "normalized_correct"]]
        .agg(["mean", "count"])
    )
    external_summary_df.columns = ["_".join(column) for column in external_summary_df.columns]
    external_summary_df = external_summary_df.reset_index()

    display(external_summary_df)
    external_scores_df.to_csv(REPORT_ROOT / "external_candidate_scores.csv", index=False)
    external_predictions_df.to_csv(REPORT_ROOT / "external_predictions.csv", index=False)
    external_summary_df.to_csv(REPORT_ROOT / "external_summary.csv", index=False)

    plt.figure(figsize=(11, 5))
    sns.barplot(
        data=external_summary_df,
        x="benchmark", y="normalized_correct_mean", hue="arm"
    )
    plt.ylim(0, 1)
    plt.title("External zero-shot benchmark accuracy")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / "external_benchmarks.png", dpi=170)
    plt.show()


## 24. Evaluate LAMBADA last-word prediction

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 24 — LAMBADA: teacher-forced last-word exact match and NLL
LAMBADA_RESULTS = []
try:
    try:
        lambada_ds = load_dataset("EleutherAI/lambada_openai", split="test")
    except Exception:
        lambada_ds = load_dataset("lambada", split="test", trust_remote_code=True)
    lambada_ds = take_limit(lambada_ds, EXTERNAL_LIMIT)

    lambada_entries = []
    for item_index, row in enumerate(lambada_ds):
        text = row["text"].strip()
        parts = text.rsplit(maxsplit=1)
        if len(parts) != 2:
            continue
        context, last_word = parts
        lambada_entries.append({
            "item_index": item_index,
            "item_id": f"lambada_{item_index}",
            "benchmark": "lambada",
            "category": "last_word",
            "context": context,
            "continuation": " " + last_word,
            "choice_index": 0,
            "label": 0,
        })

    for arm in ("R3", "M3"):
        model, _ = load_model(arm)
        scores = score_candidate_entries(model, lambada_entries)
        result = {
            "arm": arm,
            "examples": len(scores),
            "exact_match": float(scores["teacher_forced_exact"].mean()),
            "mean_target_tokens": float(scores["continuation_tokens"].mean()),
            "mean_normalized_nll": float(-scores["normalized_loglikelihood"].mean()),
        }
        LAMBADA_RESULTS.append(result)
        scores.assign(arm=arm).to_csv(
            REPORT_ROOT / f"{arm.lower()}_lambada_items.csv", index=False
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()

    lambada_summary_df = pd.DataFrame(LAMBADA_RESULTS)
    display(lambada_summary_df)
    lambada_summary_df.to_csv(REPORT_ROOT / "lambada_summary.csv", index=False)
except Exception as exc:
    lambada_summary_df = pd.DataFrame()
    print("LAMBADA could not be loaded; the other tests remain valid:", repr(exc))


## 25. Define generation and repetition metrics

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 25 — generation and repetition measurement tools
GENERATION_PROMPTS = [
    {"category": "continuation", "prompt": "The future of health care depends on"},
    {"category": "science", "prompt": "Science is important because"},
    {"category": "physical", "prompt": "If a glass falls onto a stone floor, it will probably"},
    {"category": "physical", "prompt": "If a box is placed on a table, the box is"},
    {"category": "causal", "prompt": "Maya forgot to water the plant for a month, so"},
    {"category": "temporal", "prompt": "The alarm rang before the door opened. After the door opened,"},
    {"category": "narrative", "prompt": "Alice found a locked wooden box in the garden. She"},
    {"category": "explanation", "prompt": "Rain forms when"},
    {"category": "contrast", "prompt": "Unlike a bird, a fish cannot"},
    {"category": "planning", "prompt": "To make a cup of tea, first"},
    {"category": "counterfactual", "prompt": "If the sun did not rise tomorrow,"},
    {"category": "multilingual_control", "prompt": "One day in Istanbul"},
]


@torch.inference_mode()
def generate_text(
    model: nn.Module,
    prompt: str,
    mode: str,
    max_new_tokens: int,
    seed: int,
) -> dict:
    prompt_ids = encode_text(prompt)
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    generated = []

    for _ in range(max_new_tokens):
        context = x[:, -model.config.max_seq_len:]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(context).logits[:, -1].float()
        if mode == "greedy":
            next_id = logits.argmax(-1, keepdim=True)
        elif mode == "sample":
            logits = logits / 0.8
            values, indices = torch.topk(logits, min(40, logits.size(-1)))
            filtered = torch.full_like(logits, -float("inf"))
            filtered.scatter_(1, indices, values)
            probs = torch.softmax(filtered, dim=-1)
            next_id = torch.multinomial(probs, 1, generator=generator)
        else:
            raise KeyError(mode)
        token_id = int(next_id.item())
        generated.append(token_id)
        x = torch.cat([x, next_id], dim=1)
        if EOS_ID is not None and token_id == EOS_ID:
            break

    return {
        "completion": TOKENIZER.decode(generated, skip_special_tokens=True),
        "generated_token_ids": generated,
    }


def repetition_metrics(token_ids: list[int]) -> dict:
    result = {
        "generated_tokens": len(token_ids),
        "unique_token_ratio": len(set(token_ids)) / max(1, len(token_ids)),
    }
    for n in (2, 3, 4):
        ngrams = [tuple(token_ids[i:i+n]) for i in range(max(0, len(token_ids) - n + 1))]
        distinct = len(set(ngrams)) / max(1, len(ngrams))
        result[f"distinct_{n}"] = distinct
        result[f"repeated_{n}gram_ratio"] = 1.0 - distinct
    return result


## 26. Create generations and blind A/B review files

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 23 — R3/M3 generations and blind A/B evaluation filesi

EXPECTED_GENERATION_ROWS = (
    len(GENERATION_PROMPTS)
    * 2  # R3 and M3
    * 2  # greedy and sample
)

required_columns = {
    "arm",
    "prompt_id",
    "category",
    "mode",
    "prompt",
    "completion",
}

reuse_existing_generations = (
    "generation_df" in globals()
    and isinstance(generation_df, pd.DataFrame)
    and len(generation_df) == EXPECTED_GENERATION_ROWS
    and required_columns.issubset(generation_df.columns)
)


# ============================================================
# 1. GENERATE OUTPUTS IF NEEDED
# ============================================================

if reuse_existing_generations:
    print(
        "existing generation_df is-used; "
        "modeller again will-not-be-run."
    )

else:
    GENERATION_ROWS = []

    for arm in ("R3", "M3"):
        model, _ = load_model(arm)
        model.eval()

        for prompt_id, prompt_row in enumerate(
            GENERATION_PROMPTS
        ):
            for generation_mode in (
                "greedy",
                "sample",
            ):
                generated = generate_text(
                    model=model,
                    prompt=prompt_row["prompt"],
                    mode=generation_mode,
                    max_new_tokens=GENERATION_MAX_NEW,
                    seed=SEED + int(prompt_id),
                )

                GENERATION_ROWS.append({
                    "arm": str(arm),
                    "prompt_id": int(prompt_id),
                    "category": str(
                        prompt_row["category"]
                    ),
                    "mode": str(generation_mode),
                    "prompt": str(
                        prompt_row["prompt"]
                    ),
                    "completion": str(
                        generated["completion"]
                    ),
                    **repetition_metrics(
                        generated[
                            "generated_token_ids"
                        ]
                    ),
                })

        del model

        gc.collect()
        torch.cuda.empty_cache()

    generation_df = pd.DataFrame(
        GENERATION_ROWS
    )


# NumPy numbers standard Python numbers convert.
generation_df["prompt_id"] = (
    generation_df["prompt_id"]
    .astype(int)
)


# ============================================================
# 2. SAVE GENERATIONS
# ============================================================

generation_df.to_csv(
    REPORT_ROOT / "generation_samples.csv",
    index=False,
)

generation_metrics_df = (
    generation_df
    .groupby(
        ["arm", "mode"],
        as_index=False,
    )[
        [
            "unique_token_ratio",
            "repeated_4gram_ratio",
        ]
    ]
    .mean()
)

display(generation_metrics_df)


# ============================================================
# 3. blind A/B evaluation file
# ============================================================

blind_rows = []
blind_key = []

blind_rng = random.Random(SEED)

grouped_generations = generation_df.groupby(
    ["prompt_id", "mode"],
    sort=True,
)

for (
    raw_prompt_id,
    raw_mode,
), group in grouped_generations:

    prompt_id = int(raw_prompt_id)
    generation_mode = str(raw_mode)

    records = group.to_dict(
        orient="records"
    )

    if len(records) != 2:
        raise RuntimeError(
            f"Prompt {prompt_id}, mode "
            f"{generation_mode}: "
            f"Expected outputs from two models, "
            f"{len(records)} found."
        )

    blind_rng.shuffle(records)

    blind_rows.append({
        "prompt_id": prompt_id,
        "mode": generation_mode,
        "category": str(
            records[0]["category"]
        ),
        "prompt": str(
            records[0]["prompt"]
        ),
        "completion_A": str(
            records[0]["completion"]
        ),
        "completion_B": str(
            records[1]["completion"]
        ),
        "grammar_winner": "",
        "coherence_winner": "",
        "reasoning_winner": "",
        "overall_winner": "",
        "notes": "",
    })

    blind_key.append({
        "prompt_id": prompt_id,
        "mode": generation_mode,
        "A": str(records[0]["arm"]),
        "B": str(records[1]["arm"]),
    })


blind_df = pd.DataFrame(
    blind_rows
)


# ============================================================
# 4. JSON for safe DONUSTURUCU
# ============================================================

def json_safe_value(value):
    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"{type(value).__name__} JSON'a cannot-be-converted."
    )


# ============================================================
# 5. SAVE FILES
# ============================================================

blind_review_path = (
    REPORT_ROOT
    / "blind_generation_review.csv"
)

blind_key_path = (
    REPORT_ROOT
    / "blind_generation_key.json"
)

blind_df.to_csv(
    blind_review_path,
    index=False,
)

blind_key_path.write_text(
    json.dumps(
        blind_key,
        indent=2,
        ensure_ascii=False,
        default=json_safe_value,
    ),
    encoding="utf-8",
)


# ============================================================
# 6. RESULT
# ============================================================

display(blind_df.head())

print(
    "generation examples :",
    REPORT_ROOT / "generation_samples.csv",
)

print(
    "blind evaluation:",
    blind_review_path,
)

print(
    "Gizli model anahtari:",
    blind_key_path,
)

## 27. Run paired accuracy and McNemar significance tests

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 27 — R3/M3 paired accuracy comparisonsi and McNemar exact test
def paired_accuracy_test(predictions: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    for benchmark, group in predictions.groupby("benchmark"):
        pivot = group.pivot(index="item_id", columns="arm", values=metric).dropna()
        if not {"R3", "M3"}.issubset(pivot.columns):
            continue
        r3 = pivot["R3"].astype(bool)
        m3 = pivot["M3"].astype(bool)
        r3_only = int((r3 & ~m3).sum())
        m3_only = int((~r3 & m3).sum())
        discordant = r3_only + m3_only
        p_value = (
            float(binomtest(r3_only, discordant, 0.5).pvalue)
            if discordant > 0 else 1.0
        )
        rows.append({
            "benchmark": benchmark,
            "metric": metric,
            "items": len(pivot),
            "r3_accuracy": float(r3.mean()),
            "m3_accuracy": float(m3.mean()),
            "r3_only_correct": r3_only,
            "m3_only_correct": m3_only,
            "mcnemar_exact_p": p_value,
        })
    return pd.DataFrame(rows)


all_prediction_frames = [controlled_predictions_df]
if not external_predictions_df.empty:
    all_prediction_frames.append(external_predictions_df)
paired_predictions = pd.concat(all_prediction_frames, ignore_index=True)

significance_df = pd.concat([
    paired_accuracy_test(paired_predictions, "raw_correct"),
    paired_accuracy_test(paired_predictions, "normalized_correct"),
], ignore_index=True)
display(significance_df)
significance_df.to_csv(REPORT_ROOT / "paired_accuracy_significance.csv", index=False)


## 28. Combine all results into a machine-readable report

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 28 — Combine all results in one machine-readable report
def records_or_empty(frame) -> list[dict]:
    return [] if frame is None or frame.empty else frame.to_dict("records")


final_report = {
    "metadata": {
        "seed": SEED,
        "full_eval": FULL_EVAL,
        "device": torch.cuda.get_device_name(0),
        "tokenizer": str(TOKENIZER_PATH),
        "manifest": None if MANIFEST_PATH is None else str(MANIFEST_PATH),
        "warning": (
            "Existing corpus was not benchmark-decontaminated. External benchmark "
            "scores are diagnostic, not publication-grade proof."
        ),
    },
    "checkpoint_inventory": inventory_df.to_dict("records"),
    "causal_tests": causal_df.to_dict("records"),
    "base_validation": list(BASE_VALIDATION.values()),
    "r3_ablations": records_or_empty(globals().get("r3_ablation_df", pd.DataFrame())),
    "m3_ablations": records_or_empty(globals().get("m3_ablation_df", pd.DataFrame())),
    "m3_router_diagnostics": records_or_empty(ROUTER_DIAGNOSTICS),
    "m3_expert_similarity": records_or_empty(EXPERT_SIMILARITY),
    "controlled_summary": records_or_empty(controlled_summary_df),
    "external_summary": records_or_empty(external_summary_df),
    "lambada_summary": records_or_empty(lambada_summary_df),
    "generation_metrics": records_or_empty(
        generation_df.groupby(["arm", "mode"], as_index=False)[
            ["unique_token_ratio", "repeated_4gram_ratio"]
        ].mean()
    ),
    "paired_significance": records_or_empty(significance_df),
    "external_load_errors": EXTERNAL_LOAD_ERRORS,
}

final_json_path = REPORT_ROOT / "anka_r3_m3_deep_evaluation.json"
final_json_path.write_text(
    json.dumps(
        final_report,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

summary_lines = [
    "# ANKA R3/M3 Deep Evaluation",
    "",
    "## Checkpoints",
    inventory_df.to_markdown(index=False),
    "",
    "## Corpus validation",
    (
        pd.DataFrame(BASE_VALIDATION.values()).to_markdown(index=False)
        if BASE_VALIDATION else "Corpus validation unavailable."
    ),
    "",
    "## Controlled benchmark",
    controlled_summary_df.to_markdown(index=False),
    "",
    "## External benchmark",
    (
        external_summary_df.to_markdown(index=False)
        if not external_summary_df.empty else "External benchmark unavailable."
    ),
    "",
    "## Interpretation warning",
    "The existing training corpus was not benchmark-decontaminated. Scores are diagnostic.",
]
markdown_path = REPORT_ROOT / "anka_r3_m3_deep_evaluation.md"
markdown_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("\nDEGERLENDIRME completed")
print("JSON report :", final_json_path)
print("summary report :", markdown_path)
print("blind A/B    :", REPORT_ROOT / "blind_generation_review.csv")
print("all ara CSV and grafikler:", REPORT_ROOT)
